### 주제

물류 유통량 예측 경진대회

### 데이터

train_df.csv

- index : 인덱스
- 송하인_격자공간고유번호
- 수하인 격자공간고유번호
- 택배_카테고리
- 운송장_건수

test_df.csv

- index : 인덱스
- 송하인_격자공간고유번호
- 수하인 격자공간고유번호
- 택배_카테고리

sample_submission.csv

- index : 인덱스
- 운송장_건수

### 코드 흐름

1. 라이브러리 및 데이터 로드
    - pandas, numpy
    - CatBoostRegressor
    - train_test_split
    - 평가 지표 계산용 모듈
2. 전처리
    - 결측치 처리
        - 수치형 변수→ 평균/중앙값 대체
        - 범주형 변수→ 그대로 사용(CatBoost는 인코딩 자동 처리 가능)
    - 파생 변수 생성
        - year, month, day, weekday
3. 학습/검증 데이터 분리
    - 20% 검증 데이터 분리
4. CatBoost 모델 정의
    - 깊은 트리+낮은 학습률
    
    ```python
    model = CatBoostRegressor(
        iterations=3000,
        learning_rate=0.03,
        depth=8,
        eval_metric='RMSE',
        random_seed=42,
        verbose=200
    )
    ```
    
5. 모델 학습
    
    ```python
    model.fit(
        X_train, y_train,
        eval_set=(X_valid, y_valid),
        early_stopping_rounds=200
    )
    ```
    
6. 검증 성능 확인
    - RMSE 기준 평가
    
    ```python
    from sklearn.metrics import mean_squared_error
    
    pred_valid = model.predict(X_valid)
    rmse = np.sqrt(mean_squared_error(y_valid, pred_valid))
    ```
    
7. 피처 중요도 확인
    
    ```python
    import matplotlib.pyplot as plt
    
    importances = model.get_feature_importance()
    ```
    
8. 테스트 데이터 예측 및 제출 파일 생성

⇒ CatBoostRegressor를 활용해 범주형 인코딩 없이 학습을 진행하고, early stopping과 하이퍼파라미터 튜닝을 통해 RMSE 8.4329 달성

### 코드 핵심 특징

- CatBoost의 범주형 자동 처리 기능 활용
- 비교적 깊은 트리
- 낮은 learning rate + 많은 iterations
- Early Stopping 적극 활용
- Feature Importance 확인

### 새롭게 알게 된 내용 / 어려운 내용 / 배울 점

- CatBoost의 범주형 자동 처리 기능을 활용한 점이 인상적이었다. 보통 범주형 변수를 사용할 때 One-Hot Encoding이나 Label Encoding을 먼저 수행하지만, CatBoost는 내부적으로 범주형 변수를 처리할 수 있다는 점이 인상적이었다. 덕분에 전처리 관정이 간결해지고, 과도한 차원 증가를 방지할 수 있었다.
- Early Stopping을 활용하여 과적합을 방지할 수 있었다.
- 낮은 learning rate와 많은 iteration 조합을 통해 안정적인 학습을 할 수 있었다. 여기서 depth가 모델의 복잡도를 조절하는 핵심 요소라는 점을 다시 정리할 수 있었다.
- 단순히 점수만 올리는 것이 아니라, 피처 중요도를 확인하여 어떤 변수가 중요한지 분석한 점이 인상 깊었다.
- 별도의 복잡한 전처리 없이도 CatBoost 하나만으로 상위권 성능을 달성했다는 점에서, 트리 기반 모델의 강력함과 실전 활용도를 체감할 수 있었다.